# Jeu de base

Dans ce Notebook, nous allons définir des classes pour faire tourner notre jeu de trader.

L'idée est de pouvoir simuler les actions d'un trader :
- sa stratégie
- les montants investis
- la période de temps pour "connaître" les données
- la période de temps pour réaliser sa stratégie
- ...

Voici le plan du Notebook :
1. charger les données
2. définir une classe `Account`
3. définir une classe `Game`
4. définir une classe `Strategy`
5. définir une classe `EvalGame`

Pour toutes ces classes, le code est fourni. Il faudra donc le comprendre et le tester.

Après ce Notebook, vous pourrez commencer à implémenter des stratégies, exécuter le jeu, évaluer vos stratégies, enrichir ces classes, ...


## Chargement des données

Commençons par charger tout simplement nos données.

Plus tard, vous pourrez enrichir vos données avec d'autres valeurs mobilières (des actions, des monnaies, ...). Il faudra tester vos stratégies sur plusieurs jeux de données.

In [3]:
%run -i "BaseLoader.ipynb"

loader = Loader(debug=False)

loader.loadSource(Loader.COINCODEX)

df = loader.get(Loader.COINCODEX, Loader.BTC)
btc = pd.Series(data=df['Close'])

btc.head()

Date
2010-07-17    0.0500
2010-07-18    0.0858
2010-07-19    0.0808
2010-07-20    0.0747
2010-07-21    0.0792
Name: Close, dtype: float64

## Classes de bases du jeu

Notre programme de prédiction est composée de trois classes de bases :
- une classe `Account` représentant le compte du joueur, avec son montant d'argent et son montant de valeurs mobilières (coins/actions/bonds...)
- une classe `Game` représentant l'architecture du programme exécuté pour tester notre stratégie de vente/achat
- une classe `IStrategy`, une interface, donc une classe vide qu'il faudra ré-implémenter selon les stratégies que vous voulez appliquer
- une classe `TestGame` testant une stratégie de trader  

### Account

In [4]:
class Account :
    def __init__(self, cash=100.0):
        '''
        @param cash : cash initial de l'utilisateur
        @param isPart : possible d'acheter/vendre des fractions de la valeur mobilière
        
        '''
        
        self.cash = cash
        
        self.cashInvest = self.cash 
        self.security = 0

    def buy(self, numberOfSecurity, unityPrice):
        raise NameError("Account - buy : deprecated method, use invest instead")

    def buyMax(self, unityPrice):
        raise NameError("Account - buy : deprecated method, use investAll instead")

    def sell(self, numberOfSecurity, unityPrice):
        raise NameError("Account - sell : deprecated method, use desinvest instead")

    def sellMax(self, unityPrice):
        raise NameError("Account - sell : deprecated method, use desinvestAll instead")

    def invest(self, cash, unityPrice):
        if unityPrice < 0:
            raise NameError("Account - invest : unityPrice can not be negative")
        
        if(self.cash - cash >= 0):
            self.cash = self.cash - cash
            self.security = self.security + cash / unityPrice
        else:
            raise NameError("Account - invest : not enough money")

    def investAll(self, unityPrice):
        self.invest(self.cash, unityPrice)
    
    def desinvest(self, numberOfSecurity, unityPrice):
        if unityPrice < 0:
            raise NameError("Account - desinvest : unityPrice can not be negative")
        
        if(self.security - numberOfSecurity >= 0):
            self.security = self.security - numberOfSecurity
            self.cash = self.cash + numberOfSecurity * unityPrice
        else:
            raise NameError("Account - desinvest : not enough securities")
        
    def desinvestAll(self, unityPrice):
        self.desinvest(self.security, unityPrice)

    def cashIn(self, cash):
        if cash < 0:
            raise NameError("Account - cashIn : cash can not be negative")
        self.cashInvest = self.cashInvest + cash
        self.cash = self.cash + cash

    def getTheoricalValue(self, unityPrice):
        return (self.cash + self.security * unityPrice)
    
    def getTheoricalGain(self, unityPrice):
        return self.getTheoricalValue(unityPrice) - self.cashInvest

    ''' 
    Méthode permettant d'afficher vos objets Account de manière conviviale. Cette méthode est
    automatiquement appelée quand un objet Account est donné en argument de la fonction 'print'
    '''
    def __str__(self):
        return "Cash courant: "+str(self.cash)+" & Security: "+str(self.security)+" & Cash investi: "+str(self.cashInvest)

**Questions**:
1. créer le compte d'un joueur
2. afficher le compte de ce joueur en l'affichant via la fonction `print`. D'où vient cet affichage ?
3. acheter des coins avec les fonctions `invest` ou `investAll`
4. vender des coins avec les fonctions `desinvest` ou `desinvestAll`
5. dépenser tout votre argent
6. afficher le montant de votre portefeuille (celui-ci dépend de la valeur actuelle de la valeur mobilière - indiquer la valeur que vous voulez)
7. afficher vos gains (le gain dépend de la valeur actuelle de la valeur mobilière - indiquer la valeur que vous voulez)

Pour chaque questions, tester le résultat en affichant les montants restants et le nombre de valeurs mobilières disponibles

In [5]:
Philippe = Account(1000)
print(Philippe)

#L'affichage vient de la méthode de classe __str__() de la classe Account.
Philippe.invest(500, 100)
Philippe.desinvest(2, 100)
Philippe.investAll(100)
print(Philippe)

gain = Philippe.getTheoricalGain(120)
print(f"Gain théorique : {gain}€")

Cash courant: 1000 & Security: 0 & Cash investi: 1000
Cash courant: 0 & Security: 10.0 & Cash investi: 1000
Gain théorique : 200.0€


### Game & Strategy

La classe `Game` prend une série temporelle en entrée et se configure avec une stratégie. Le jeu comprend des actions de base :
- acheter `INVEST`
- vendre `DESINVEST`
- ne rien faire `STILL`

Voici les tâches réalisées quand le jeu s'exécute (méthode `run`):
1. lecture d'une valeur de la série temporelle
2. sélection d'une action en fonction de la stratégie testée : la stratégie prend une décision en fonction de la valeur lue et retourne un couple (action, montant de valeur mobilière), par exemple `(BUY, 150)` pour acheter 150 valeurs mobilières
3. application de l'action sur le compte du tradeur

Voici, ci-dessous, la classe `Game`. En résumé, cette classe contient :
- les actions `INVEST`, `DESINVEST`, `STILL`
- un constructeur prenant en paramètre la série temporelle, un `Account` et une `Strategy` de trading
- deux setteurs permettant d'initialiser (ou mettre à jour) le `Account` et la `Strategy` du jeu
- la méthode `run` parcourant une à une la série temporelle, récupérant l'action (action, montant) de la stratégie à partir de la valeur du jour. L'action est appliquée sur le compte
- une méthode `getTheoricalValue` retournant la valeur théorique du `Account` (théorique car cela suppose que l'on revende toutes nos valeurs mobilières au cours du jour)
- une méthode `getTheoricalGain` retournant le gain théorique, soustrayant le montant initial à notre valeur théorique


In [ ]:

class Game :

    '''
    Définition des variables appartenant à la classe, et non pas aux objets
    '''
    BUY = 0
    SELL = 1
    STILL = 2
    INVEST = 3
    DESINVEST = 4
    INVEST_ALL = 5
    DESINVEST_ALL = 6
    CASH_IN = 7

    ACTIONS = ["BUY", "SELL", "STILL", "INVEST", "DESINVEST", "INVEST_ALL", "DESINVEST_ALL", "CASH_IN"]

    @staticmethod
    def printAction(action):
        return Game.ACTIONS[action]

    def __init__(self, prices, account=None, strategy=None, data=None):
        '''
        @param prices : prix de la valeur mobilière de type pandas.Series
        @param account: instance de la classe Account
        @param strategy: instance de la classe Strategy
        @param data: données supplémentaires pour la stratégie
        '''
        self.dates = prices.index
        print(self.dates)
        self.prices = prices.tolist()

        self.data = data
        if self.data is None:
            self.data = self.prices
        else:
            self.data = data.to_numpy()

        self.account = account
        self.strategy = strategy
        self.clock = 0
        
    def setStrategy(self, strategy):
        self.strategy = strategy

    def setAccount(self, account):
        self.account = account
        self.clock = 0

    def run(self, debug=False):

        if(self.account is None):
            raise NameError("Account is not set")
        
        if(self.strategy is None):
            raise NameError("Strategy is not set")
        
        for i in range(0, len(self.data)):

            unityPrice = self.prices[i]
            unityData = self.data[i]
            
            (action, amount) = self.strategy.getAction(self, unityPrice, self.account, self.dates[self.clock], unityData, debug)
            print(action, amount)

            if debug:
                print("---", Game.printAction(action), amount, "securities at clock ", self.clock)

            try:
                if(action == Game.BUY):
                    self.account.buy(amount, unityPrice)
                elif(action == Game.SELL):
                    self.account.sell(amount, unityPrice)
                elif(action == Game.INVEST):
                    self.account.invest(amount, unityPrice)
                elif(action == Game.DESINVEST):
                    self.account.desinvest(amount, unityPrice)
                elif(action == Game.INVEST_ALL):
                    self.account.investAll(unityPrice)
                elif(action == Game.DESINVEST_ALL):
                    self.account.desinvestAll(unityPrice)
                elif(action == Game.CASH_IN):
                    self.account.cashIn(amount)
                else:
                    pass
            except NameError as error:
                    print(error)

            if debug:
                print(self.account)
                
            self.clock = self.clock+1

    def getTheoricalValue(self):
        return self.account.getTheoricalValue(self.prices[-1])
    
    def getTheoricalGain(self):
        return self.account.getTheoricalGain(self.prices[-1])

    def __str__(self):
        return "Game after "+str(self.clock)+"/"+str(len(self.prices))+" steps - "+str(self.account)
    

**Questions**: 
1. lire la classe `Game`
2. instancier un objet `game`, à partir de la classe `Game`
3. exécuter la méthode `run`. Que se passe-t-il ?
4. initialiser le `Account` de `game` avec l'objet `account` créé précédemment
5. exécuter la méthode `run`. Que se passe-t-il ?
6. continuer à lire le Notebook, pour découvrir la classe `Strategy`


In [7]:
# Instanciation de Game
game = Game(btc, Philippe)
game.run()

# 3. Pas de compte passé en argument
# 4. Pas de stratégie passée en argument

DatetimeIndex(['2010-07-17', '2010-07-18', '2010-07-19', '2010-07-20',
               '2010-07-21', '2010-07-22', '2010-07-23', '2010-07-24',
               '2010-07-25', '2010-07-26',
               ...
               '2025-10-25', '2025-10-26', '2025-10-27', '2025-10-28',
               '2025-10-29', '2025-10-30', '2025-10-31', '2025-11-01',
               '2025-11-02', '2025-11-03'],
              dtype='datetime64[s]', name='Date', length=5589, freq=None)


NameError: Strategy is not set

Pour exécuter un jeu, il nous faut donc un instance de `Strategy`. Voici ci-dessous la classe de base `Strategy` : cette classe est un squelette que toutes vos classes stratégies vont devoir respecter. 
L'utilité de respecter ce squelette est de pouvoir utiliser toutes vos stratégies avec la classe `Game` précédente.

Cette classe est définie par :
- un constructeur, dans lequel on pourra initialiser des variables, suivant notre stratégie. Deux paramètres sont présents pour le moment, le nom de la stratégie, et si la stratégie est aléatoire ou non (`False` indique que la stratégie n'est pas aléatoire et donc une seule passe de test est nécessaire pour l'évaluer)
- une méthode `getAction` prenant en paramètre un objet `Game`, un prix actuel `unityPrice`, la date de ce prix et un paramètre `debug` indiquant si il faut afficher des messages quand cette fonction est appelée

In [ ]:
class IStrategy :

    def __init__(self, name="IStrategy", _isRandom=False):
        self.name = name
        self._isRandom = _isRandom

    def getAction(self, game, unityPrice, account, date, unityData=None, debug=False):
        '''
        @return (action, amountOfSecurity)
        '''
        pass

`IStrategy` est une classe de base, utile ici simplement pour vous indiquer quelle tête vos futures stratégies doivent avoir pour fonctionner avec `Game`. Vos stratégies doivent donc avoir la même tête, c'est à dire, les mêmes méthodes.

Pour être sûr d'avoir la même tête, il faut que vos stratégies héritent de la classe `IStrategy`. Et voici comment faire :

In [ ]:
class MaNouvelleStrategy(IStrategy):

    def __init__(self):
        super().__init__("MaNouvelleStrategie", False)

    def getAction(self, game, unityPrice, account, date, unityData=None, debug=False):
        '''
        implementation de votre algorithme ici

        vous connaissez le prix du jour unityPrice, la date du jour

        vous avez accès aux prix avec le paramètre:
        unityPrice

        vous avez accès au compte avec le paramètre:
        account
        
        donc il vous reste plus qu'à prendre une décision
        et retourner votre action à réaliser sous forme de tuple (Game.BUY, 10) pour acheter 10 valeurs mobilières par exemple
        Ou bien (Game.SELL, 8) pour en vendre 8
        '''
        pass
        

Notez la première ligne de code de la cellule précédente `class MaNouvelleStrategy(IStrategy)`, qui indique que la classe `MaNouvelleStrategy` hérite de la classe `IStrategy`. Toute vos stratégies devront faire pareil.

**Questions** :
1. ouvrir un Notebook `Strategies.ipynb`
2. lire et comprendre la stratégie `StrategyHelper` de ce fichier
3. charger le Notebook `Strategies.ipynb` dans ce Notebook
4. dans le fichier `Strategies.ipynb`, ajouter une classe stratégie `StrategyLongTerm`. Bien entendu, cette classe hérite de la classe `IStrategy` et vérifiera son squelette. La stratégie `LongTerm` consiste à dépenser tout votre argent à une date décidée, à les garder, puis tout revendre aujourd'hui
5. dans le fichier `Strategies.ipynb`, ajouter une classe stratégie `StrategyRandom`. Bien entendu, cette classe hérite de la classe `IStrategy` et vérifiera son squelette. La stratégie `Random` consiste à choisir au hasard l'action réalisée, le montant d'achat ou de vente (montant dans les bornes du cash et des valeur mobilières disponibles)
6. dans le fichier `Strategies.ipynb`, ajouter une classe stratégie `StrategyOracle`. La stratégie `Oracle` est la meilleure stratégie possible, et sous entend que l'on puisse déterminer le futur : on vend au plus haut et on rachète quand on est au plus bas. Cela revient à détecter les minimum et maximum locaux. Toutes les stratégies que nous implémenterons dans la suite pourront ainsi être comparées à la stratégie `Oracle`. 

Chargez ici votre fichier stratégies :

In [84]:
# chargement du Notebook Strategies.ipynb
%run -i "Strategies.ipynb"

Testez maintenant vos stratégies, voici un squelette d'exemple :

In [112]:

# création de l'objet strategy
strategy = StrategyRandom()

# création de Account
account = Account()

# création de Game
game = Game(btc, account, strategy)

# configuration de game avec account et strategy


# execution du jeu
game.run()
value = game.getTheoricalValue()

print(game)
print(value)

DatetimeIndex(['2010-07-17', '2010-07-18', '2010-07-19', '2010-07-20',
               '2010-07-21', '2010-07-22', '2010-07-23', '2010-07-24',
               '2010-07-25', '2010-07-26',
               ...
               '2025-10-25', '2025-10-26', '2025-10-27', '2025-10-28',
               '2025-10-29', '2025-10-30', '2025-10-31', '2025-11-01',
               '2025-11-02', '2025-11-03'],
              dtype='datetime64[s]', name='Date', length=5589, freq=None)
6 0
4 0.0
6 0
6 0
3 92.11756485139894
4 457.0289866219623
6 0
6 0
4 0.0
6 0
3 44.8739317009789
4 571.3126739385897
5 0
4 80.79459879695702
3 2.733946099411715
5 0
6 0
6 0
5 0
5 0
5 0
5 0
6 0
4 0.0
6 0
4 0.0
6 0
3 35.69947627435753
6 0
5 0
5 0
4 895.4438549954965
4 186.5507499193315
5 0
4 632.9846902737663
5 0
4 271.4785439414705
5 0
4 824.4952091434901
4 286.43162392227043
6 0
3 33.160791873260706
3 20.195680219350535
5 0
5 0
5 0
5 0
3 0.0
6 0
5 0
6 0
6 0
6 0
3 4.166848037085715
3 62.99351769918644
6 0
3 32.97031193530538
5 0
5 0


## EvalGame

Il nous reste maintenant à présenter la classe `EvalGame` : celle-ci sert à exécuter automatiquement un nombre X de fois un jeu avec une stratégie associée. Certaines stratégies ayant des comportements stochastique (_Se dit de phénomènes qui, partiellement, relèvent du hasard et qui font l'objet d'une analyse statistique_), chaque exécution donnera un résultat différent. Pour les évaluer, il est nécessaire d'aggréger le résultat de plusieurs exécutions.

Attention, la version de la classe présentée est plus simple que celle nous utiliserons.

Par exemple :
- notre stratégie `Random` rend tous le temps un résultat différent. Il est nécessaire de calculer le gain moyen de cette stratégie pour l'évaluer
- notre stratégie `Oracle` est par contre déterministe. Chaque itération génére le même résultat

Une courte description de la classe 'EvalGame':
- un constructeur prenant en paramètre une série temporelle, un montant initial de cash pour le `Account` et une `Strategy`
- une méthode `setStrategy` pour initialiser/mettre à jour la stratégie du jeu
- une méthode `eval` prenant le nombre d'execution du `Game` et un paramètre `debug` indiquant si il faut afficher les actions réalisées par le jeu
- la méthode `eval` réalise `iteration ` fois :
    1. création d'un `Account`
    2. exécution du jeu avec sa stratégie
    3. stockage du gain théorique (si l'on revendait toutes nos valeurs mobilières à la fin de la série temporelle)
- une méthode `getResults` retournant la liste des gains de chaque itération

In [38]:
class EvalGame:

    def __init__(self, serie, initialAmount=100, strategy=None):
        self.game = Game(serie)
        self.game.setStrategy(strategy)
        self.initialAmount = initialAmount        
        self.results = []

    def setStrategy(self, strategy):
        self.game.setStrategy(strategy)

    def eval(self, iteration=1000, debug=False):
        
        self.results = []

        for i in range(0, iteration):
            account = Account(self.initialAmount)
            
            if(debug):
                print("init account:", account)
            self.game.setAccount(account)
            self.game.run(debug=debug)

            score = self.game.getTheoricalGain()
            self.results.append(score)

            if debug:
                print("score[", i, "]: ", score)    

    def getResults(self):
        return self.results

    def __str__(self):
        return "Avg:"+str(np.mean(self.results))+" Min: "+str(np.min(self.results))+" Max:"+str(np.max(self.results))

**Questions** :
1. instancier un objet `eval` à partir de la classe `EvalGame`, en utilisant les données précédemment chargées et la `StrategyRandom`
2. exécuter le jeu avec **100** itérations et afficher le résultat
3. exécuter le jeu avec **1000** itérations et afficher le résultat
3. afficher, pour les 2 tests précédents, un histogramme avec la fonction de densité. Pour ce faire, rechercher sur Google avec les mots clés suivant `seaborn hist density` (pour la densité, regarder le paramètre `kde`)
4. quel est le gain moyen avec une stratégie aléatoire ?
5. tester votre objet `eval` en lui donnant la `StrategyOracle`. Obtenez vous bien le même résultat à chaque itération ? avez vous trouvé facile de réaliser un autre test en changeant de stratégie ?

In [ ]:
# création de eval 
eval = EvalGame(btc, strategy=StrategyRandom)

# evaluation de la stratégie
eval.eval(iteration = 5, debug = False)

print(eval)


DatetimeIndex(['2010-07-17', '2010-07-18', '2010-07-19', '2010-07-20',
               '2010-07-21', '2010-07-22', '2010-07-23', '2010-07-24',
               '2010-07-25', '2010-07-26',
               ...
               '2025-10-25', '2025-10-26', '2025-10-27', '2025-10-28',
               '2025-10-29', '2025-10-30', '2025-10-31', '2025-11-01',
               '2025-11-02', '2025-11-03'],
              dtype='datetime64[s]', name='Date', length=5589, freq=None)


NameError: Strategy is not set

## Conclusion

Félicitation !

Vous êtes maintenant prêt pour implémenter vos propres stratégies !

Vous pouvez continuer avec le notebook `HowToStart.ipynb`, reprenant les principaux points vu ici, mais en approfondissant les classes de bases fournies pour le projet.
